# CS156 Session 8: Gradients

Backpropagation by hand on the sigmoid neuron from CS231n, the vectorized version, automatic gradients with JAX, and gradient descent.

My written answers are in `session8_workbook.md`.

Notes on this notebook:
- Cells marked "added" are small additions to the workbook code (print statements and a final call).
- The JAX cells are included exactly as in the workbook but were **not run** here, because JAX isn't available in my environment. Run them in Colab or locally to get their outputs.

## Exercises 3.1 and 3.2: backward pass for the sigmoid neuron

Every derivative is with respect to the final output `invert`. The forward-pass values are stored so the backward pass can reuse them.

In [1]:
import numpy as np

# Forward pass
w0, w1, w2 = 2.00, -3.00, -3.00
x0, x1 = -1.00, -2.00

mul0 = w0 * x0  # -2.00
mul1 =  w1 * x1  # 6.00
sum01 = mul0 + mul1  # 4.00
sum012 = sum01 + w2  # 1.00

neg = -sum012  # -1.00
exp = np.exp(neg)  # 0.37
plus1 = exp + 1  # 1.37
invert = 1/plus1  # 0.73


# Backward pass
# Remember *all* derivatives are with respect to the final function invert
d_invert = 1.00  # this is d_invertd_invert, but we shorten to d_invert
# Here's one completed for you :)
d_plus1 = d_invert * (-1/plus1**2)  # this is d_invertd_plus_1, but we shorten to d_plus_1
d_exp = d_plus1
d_neg = d_exp * exp
d_sum012 = d_neg * -1

# Notice for gates with multple inputs, when going backward
# they are responsible for derivatives of multiple values!
d_w2, d_sum01 = d_sum012, d_sum012
d_mul0, d_mul1 = d_sum012 * 1, d_sum012 * 1

d_w1, d_x1 = d_mul1 * x1, d_mul1 * w1
d_w0, d_x0 = d_mul0 * x0, d_mul0 * w0

# added: print the results to compare with the red numbers in the diagram
print("forward output:", invert)
print("d_w0, d_w1, d_w2:", d_w0, d_w1, d_w2)
print("d_x0, d_x1:", d_x0, d_x1)

forward output: 0.7310585786300049
d_w0, d_w1, d_w2: -0.19661193324148188 -0.39322386648296376 0.19661193324148188
d_x0, d_x1: 0.39322386648296376 -0.5898357997244457


## Calculus derivation

For f(w, x) = w^T x, the worked example gives df/dx = w. My derivation of df/dw = x is in `session8_workbook.md`.

## Vectorized backward pass

Here `w` and `x` are vectors. The multiply and add are fused into the matrix product `w.T @ x`, so the gradients from the derivation above are used directly: `d_w = d_dot * x` and `d_x = d_dot * w`.

In [2]:
# Forward pass
w = np.array([2.00, -3.00])
b = -3.00

x = np.array([-1.00, -2.00])

dot = w.T @ x  # 4.00
summed = dot + b  # 1.00

neg = -summed  # -1.00
exp = np.exp(neg)  # 0.37
plus1 = exp + 1  # 1.37
invert = 1/plus1  # 0.73


# Backward pass
d_invert = 1.00
d_plus1 = d_invert * (-1/plus1**2)
d_exp = d_plus1 * 1
d_neg = d_exp * exp
d_summed = d_neg * -1   # renamed from `summed` so the forward value is not overwritten

d_b, d_dot =  d_summed * 1, d_summed * 1
# Be careful, the next two are vectors!
d_w = d_dot * x
d_x = d_dot * w

# added: print the results
print("forward output:", invert)
print("d_b:", d_b)
print("d_w:", d_w)
print("d_x:", d_x)

forward output: 0.7310585786300049
d_b: 0.19661193324148188
d_w: [-0.19661193 -0.39322387]
d_x: [ 0.39322387 -0.5898358 ]


## Preview: automatic gradients with JAX (not run here)

In [3]:
%%capture
!python -m pip install -U --no-cache-dir "jax==0.4.13" "jaxlib==0.4.13" -f https://storage.googleapis.com/jax-releases/jax_releases.html

In [4]:
import jax
import jax.numpy as jnp  # JAX has a version of NumPy that acts...just like NumPy!

# We first define a function for *just* the forward pass
def sigmoid_example(params, x):
    dot = params['w'].T @ x  # 4.00
    summed = dot + params['b']  # 1.00

    neg = -summed  # -1.00
    exp = jnp.exp(neg)  # 0.37
    plus1 = exp + 1  # 1.37
    invert = 1/plus1  # 0.73

    return invert

In [5]:
params = {'w': np.array([2.00, -3.00]), 'b': -3.00}  # We define our parameters as a dict.
x = np.array([-1.00, -2.00])

sigmoid_example(params, x)  # We get the expected value of 0.73 back.

Array(0.7310586, dtype=float32)

In [6]:
grad_sigmoid_example = jax.grad(sigmoid_example)
grad_sigmoid_example(params, x)

{'b': Array(0.19661197, dtype=float32, weak_type=True),
 'w': Array([-0.19661197, -0.39322394], dtype=float32)}

## Extension: gradient descent for 1000 iterations

Goal: find weights that push the output of the sigmoid example toward 0. The manual backward pass is wrapped in a function of the weights, and the weights are updated after every iteration.

In [7]:
# Best wishes!
import numpy as np

def forward(w, b, x):
    dot = w @ x
    summed = dot + b
    neg = -summed
    exp = np.exp(neg)
    plus1 = exp + 1
    invert = 1/plus1
    cache = (x, exp, invert)          # save what the backward pass needs
    return invert, cache

def backward(cache):
    x, exp, invert = cache
    d_invert = 1.0
    d_plus1 = d_invert * (-1/(exp+1)**2)
    d_exp = d_plus1 * 1
    d_neg = d_exp * exp
    d_summed = d_neg * -1
    d_b = d_summed
    d_dot = d_summed
    d_w = d_dot * x                    # df/dw = x, from the earlier exercise
    return d_w, d_b

def gradient_descent(w, b, x, n_iters=1000, step_size=1.0):
    for i in range(n_iters):
        output, cache = forward(w, b, x)
        d_w, d_b = backward(cache)
        w = w - step_size * d_w        # gradient descent update
        b = b - step_size * d_b
    return w, b

Added: run the gradient descent and compare the output before and after.

In [8]:
w_start, b_start = np.array([2.00, -3.00]), -3.00
x_in = np.array([-1.00, -2.00])

w_final, b_final = gradient_descent(w_start, b_start, x_in, n_iters=1000, step_size=1.0)

print("output before:", forward(w_start, b_start, x_in)[0])
print("output after 1000 iterations:", forward(w_final, b_final, x_in)[0])
print("final w:", w_final, "final b:", b_final)

output before: 0.7310585786300049
output after 1000 iterations: 0.00016664123727299493
final w: [3.61658345 0.23316689] final b: -4.616583446848287
